# Topic Modeling
### Bert Sentence Embedding, LDA, and K-means Clustering

#### Concatenate the sentence embeddings with LDA, use an autoencoder to get the latent representation and perform K-means to perform clustering on the data

#### After the sentences have been assigned to their clusters, we can extract the candidate topics for each cluster using tokens of it's sentences

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
% cd /content/drive/My Drive/Thesis_colab

In [ ]:
! pip install transformers
! pip install pandas
! pip install torch
! pip install nltk
! pip install gensim
! pip install matplotlib
! pip install umap-learn
! pip install wordcloud
! pip install sklearn
! pip install tqdm
! pip install stop_words
! pip install xlrd
! pip install langdetect
! pip install openpyxl

In [1]:
import nltk
import numpy as np
import statistics
from sklearn.decomposition import PCA

In [ ]:
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

### 1. Reading and preprocessing the data

In [ ]:
from utilities.document_preprocessing import preprocess_documents

#### The preprocess_document function reads the sentences from the file and generates tokens from each sentences.

In [ ]:
# Input file
file = "data/PMP_Skills_Matching_Pilot_2018_2019_small.xlsx"

In [2]:
Verbose = False

In [3]:

#read the previously processed data
import pandas as pd
data = pd.read_pickle("output/preprocessed_data/use_this_data.pkl")
sentences = list(data['sentences'])
token_lists = list(data['token_list'])
ids = list(data['id'])
num_docs = len(ids)

In [4]:
len(ids)

205

In [ ]:
sentences, token_lists = preprocess_documents(file, verbose = Verbose)

### 2. Topic Modeling

In [5]:
from models.topic_model import Topic_Model

#### The topic_Model class does the follow
- Extract Bert sentence embeddings for all sentences (Local features)
- get topic distribution for each sentence using LDA
- concatenate both and learn the latent representation using an auto-encoder
- cluster the latent representation into K clusters

In [6]:
# define the numbe of clusters
k = 3

In [7]:
Verbose = False

In [8]:
# Define the topic model object
TopicModelObject = Topic_Model(k = k, verbose = Verbose, embedding = 'bert', 
                               saved_embedding_dir = "output/preprocessed_data/bert_embeddings.json", embedding_batch_size = 100)

In [9]:
# Fit the topic model
TopicModelObject.fit(sentences, token_lists)

Getting vector representations for LDA 
Getting vector representations for LDA. Done! 


 Getting vector representations for bert 
(205, 3)
(205, 1024)
Fitting Autoencoder
cpu
Epoch 1 of 5, Train Loss: 0.129
Epoch 2 of 5, Train Loss: 0.149
Epoch 3 of 5, Train Loss: 0.082
Epoch 4 of 5, Train Loss: 0.084
Epoch 5 of 5, Train Loss: 0.084
Fitting Autoencoder Done!
Clustering the latent representations
Clustering. Done!


### 3. Analysis
#### Some analysis on the clustered objectives

In [10]:

if TopicModelObject.method == 'LDA':
    feature_vector = TopicModelObject.vec[TopicModelObject.method]
    # for each sentence, find out the cluster it belongs to
    cluster_label_per_document = np.argmax(np.array(lda_feature), axis=1)
else:
    cluster_label_per_document = np.array(TopicModelObject.cluster_model.labels_)
    feature_vector = TopicModelObject.vec[TopicModelObject.method]
    feature_vector = feature_vector.numpy()
    

In [11]:
unique_ids = list(set(ids))
num_employee = len(unique_ids)

In [12]:
#create a dataframe to store the frequence per cluster for each employee. 
df_id_cluster = pd.DataFrame(0,
                  index=pd.Index(unique_ids, name='Employee id'),
                  columns=pd.Index([i for i in range(k)], name='Clusters'))
#There are multiple entries per employee so an employee can belong to multiple clusters
for i, _id in enumerate(ids):
    df_id_cluster[cluster_label_per_document[i]][_id] += 1
    
employee_cluster_frequency = df_id_cluster.values.tolist()
employee_cluster_frequency

[[0, 0, 7], [66, 72, 24], [12, 6, 18]]

#### Dimensionality Reduction
##### Perform Dimensionality reduction on the learned features, merged all the vectors of each employee into one by taking average and assign the class label based on argmax on label counts

In [27]:
# Create an employee based analysis. Compress all learnt vector into 2D space
average_representation = pd.DataFrame(None,
                  index=pd.Index(unique_ids, name='Employee id'),
                  columns=pd.Index(['feature_vector','cluster','feature_vector_2d','feature_vector_3d', 'color'],
                                   name='Compressed_analysis'))


In [28]:
average_representation

Compressed_analysis,feature_vector,cluster,feature_vector_2d,feature_vector_3d,color
Employee id,,,,,
*100002,NaN,NaN,NaN,NaN,NaN
*100000,NaN,NaN,NaN,NaN,NaN
*100001,NaN,NaN,NaN,NaN,NaN


In [29]:
# get the indices of all objectives/documents/sentence for each employee
employee_objective_indices = dict()
for employee in unique_ids:
    employee_indices = [i for i, e in enumerate(ids) if (ids[i] == employee) ]
    employee_objective_indices[employee] = employee_indices

In [30]:
# take average of all feature vectors belonging to an employee
for employee in unique_ids:
    #get all feature vectors belonging to an employee
    feature_vector_per_employee = feature_vector[employee_objective_indices[employee]]
    # get label for all feature vectors belonging to that employee
    clusters_per_employee = cluster_label_per_document[employee_objective_indices[employee]]  
    
    #take mean of all feature vectors
    average_representation['feature_vector'][employee] = np.mean(feature_vector_per_employee, axis=0)
    
    # pick the label with most occurance for that employee
    average_representation['cluster'][employee] = statistics.mode(clusters_per_employee) 
    

In [31]:
#converting from array of array to 2d array
feature_vector_ = np.stack(average_representation['feature_vector'].to_numpy())

In [32]:
from sklearn.manifold import TSNE

In [33]:
# reduce dimension of feature vectors into 2
feature_vector_2d = TSNE(n_components=2).fit_transform(feature_vector_)
average_representation['feature_vector_2d'] = list(feature_vector_2d)
average_representation

Compressed_analysis,feature_vector,cluster,feature_vector_2d,feature_vector_3d,color
Employee id,,,,,
*100002,"[0.0, 0.0, 0.23071992, 0.06455507, 0.0, 0.0528...",2,"[1495.9574, -105.389755]",NaN,NaN
*100000,"[0.23087558, 0.09068832, 0.95538634, 0.2492525...",1,"[-2201.8347, 2877.598]",NaN,NaN
*100001,"[0.109171815, 0.057299837, 0.72364587, 0.15413...",2,"[-1420.1658, 101.000496]",NaN,NaN


In [34]:
# reduce dimension of feature vectors into 3
feature_vector_3d = TSNE(n_components=3).fit_transform(feature_vector_)
average_representation['feature_vector_3d'] = list(feature_vector_3d)
average_representation

Compressed_analysis,feature_vector,cluster,feature_vector_2d,feature_vector_3d,color
Employee id,,,,,
*100002,"[0.0, 0.0, 0.23071992, 0.06455507, 0.0, 0.0528...",2,"[1495.9574, -105.389755]","[-356.08426, 412.45303, -225.89876]",NaN
*100000,"[0.23087558, 0.09068832, 0.95538634, 0.2492525...",1,"[-2201.8347, 2877.598]","[902.87067, 273.45163, 415.50458]",NaN
*100001,"[0.109171815, 0.057299837, 0.72364587, 0.15413...",2,"[-1420.1658, 101.000496]","[-547.37537, -686.07666, -189.55934]",NaN


#### Visualization feature vectors for each employee after dimensionality reduction in 2d and 3d space

In [35]:
# assigning a color to each cluster for representation
cluster_color_dict = {0:'blue', 1:'red', 2:'green', 3:'grey', 4:'pink', 5:'yellow', 6:'orange',
                      7:'pink', 8:'black', 9:'cyan'}

color = [cluster_color_dict[average_representation['cluster'][emp]] for emp in average_representation.index.values]
average_representation['color'] = color
average_representation

Compressed_analysis,feature_vector,cluster,feature_vector_2d,feature_vector_3d,color
Employee id,,,,,
*100002,"[0.0, 0.0, 0.23071992, 0.06455507, 0.0, 0.0528...",2,"[1495.9574, -105.389755]","[-356.08426, 412.45303, -225.89876]",green
*100000,"[0.23087558, 0.09068832, 0.95538634, 0.2492525...",1,"[-2201.8347, 2877.598]","[902.87067, 273.45163, 415.50458]",red
*100001,"[0.109171815, 0.057299837, 0.72364587, 0.15413...",2,"[-1420.1658, 101.000496]","[-547.37537, -686.07666, -189.55934]",green


In [43]:
import plotly.express as px

In [49]:
fig = px.scatter(x=feature_vector_2d[:, 0],
                 y=feature_vector_2d[:, 1],
                color=average_representation['color'],
                hover_name = average_representation.index.values,
                width=1000, height=500)
fig.show()


## Network Visualization using pyviz

In [ ]:
from pyvis.network import Network
import networkx as nx

In [ ]:
net = Network("500px", "500px")

In [ ]:
# Create all cluster nodes

#cluster ids
cluster_node_ids = ["cluster_" + str(i+1) for i in range(k)]
print (cluster_node_ids)

#size of the cluster nodes
cluster_node_size = 1000

cluster_sizes = [cluster_node_size for _ in range(k)]
x = [40+i for i in range(k)]
y = [40+i for i in range(k)]
color = ["#dd4b39" for i in range(k)]

#main topics for each cluster
cluster_titles = []
for key, value in TopicModelObject.get_top_words_lda(top=10).items():
    cluster_title = ', '.join([item[0] for item in value])
    cluster_titles.append(cluster_title)

print(cluster_titles)


net.add_nodes(cluster_node_ids, value= cluster_sizes, 
              title=cluster_titles, 
              x=x, y=y, 
              label=cluster_node_ids, 
              color=color)


In [ ]:
# create all employee nodes

#employee ids
employee_node_ids = list(set(ids))

#size of the employee nodes
employee_node_size = 200

employee_node_sizes = [employee_node_size for _ in range(len(employee_node_ids))]
x = [10+i for i in range(len(employee_node_ids))]
y = [40+i for i in range(len(employee_node_ids))]
color = ["#162347" for i in range(len(employee_node_ids))]
 



# get all objective indices for each employee
employee_objective_indices = dict()
for employee in employee_node_ids:
    employee_indices = [i for i, e in enumerate(ids) if (ids[i] == employee) ]
    employee_objective_indices[employee] = employee_indices
    

#All objectives belong to an employee are concatenated
employee_titles = []
np_sentences = np.array(sentences)
for key, value in employee_objective_indices.items():
    employee_title = '. '.join([objective for objective in np_sentences[value]])
    
    employee_titles.append(employee_title)

    

net.add_nodes(employee_node_ids[:subset], value=employee_node_sizes[:subset], 
              title=employee_titles[:subset], 
              x=x[:subset], y=y[:subset], 
              label=employee_node_ids[:subset], 
              color=color[:subset])

In [ ]:
# create all edges between employees and clusters
for emp,employee_cluster in enumerate(employee_cluster_frequency):
    for cl, cluster in enumerate(employee_cluster):
        
        # If the employee belongs to that cluster
        if cluster > 0:
            
            #print (employee_node_ids[emp],cluster_node_ids[cl])
            net.add_edge(employee_node_ids[emp],cluster_node_ids[cl],weight=float(cluster/num_docs))
            

In [ ]:
net.show_buttons(filter_=['physics'])

In [ ]:
# visualize the network
net.show("example.html")

## Visualization: Word Cloud

In [ ]:
from utilities.visualizations import *
%matplotlib inline

### Visualizations for LDA only

In [ ]:
lda_class_distribution = TopicModelObject.vec["LDA"]
cluster_label_per_document = np.argmax(lda_class_distribution, axis=1)
print (cluster_label_per_document.shape)

In [ ]:
# To get the top words for each topic, this method only uses LDA, doesn't utilize Bert or K mean clustering
TopicModelObject.get_top_words_lda(top=10)

In [ ]:
for k , cluster_word_distribution in TopicModelObject.get_top_words_lda(top=100).items():
    get_wordcloud(TopicModelObject, word_distribution = cluster_word_distribution, topic = k )

### Visualizations for LDA + Embeddings
#### Only run when embeddings have been used

In [ ]:
labels = set(TopicModelObject.cluster_model.labels_)
for label in labels:
    index = []
    for idx, l in enumerate(TopicModelObject.cluster_model.labels_):
        if l == label:
            index.append(idx)
    print ('\n \nSentences belong to cluster ', str (label+1), ' are: \n ')
    for i in index:
        #print (sentences[i])
        continue

#### Topic Modeling for 1-gram

In [ ]:
# visualize and save img 
#visualize(TopicModelObject)
for i in range(TopicModelObject.k):
    get_wordcloud(TopicModelObject, token_lists, i)

In [ ]:
from utilities.ngram import generate_n_gram

#### Topic Modeling for 1-gram using n-gram functions

In [ ]:
for i in range(TopicModelObject.k):
    ngram_tokens_list = generate_n_gram(sentences, n=1, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=False)
    
    ngram_token_count = dict()
    for sentence in ngram_tokens_list:
        for feature in sentence:
            if feature in ngram_token_count.keys():
                ngram_token_count[feature] += 1
            else:
                ngram_token_count[feature] = 1    
    
    get_wordcloud(TopicModelObject, topic=i, word_count_dict = ngram_token_count)

#### Topic Modeling for 2-gram

In [ ]:
for i in range(TopicModelObject.k):
    ngram_tokens_list = generate_n_gram(sentences, n=4, separator = '-' ,
                                        keep_stopwords= True, stem=False, keep_nouns_only=False)
    
    ngram_token_count = dict()
    for sentence in ngram_tokens_list:
        for feature in sentence:
            if feature in ngram_token_count.keys():
                ngram_token_count[feature] += 1
            else:
                ngram_token_count[feature] = 1    
    
    get_wordcloud(TopicModelObject, topic=i, word_count_dict = ngram_token_count)

#### Topic Modeling for 3-gram

In [ ]:
from utilities.ngram import generate_n_gram

In [ ]:
for i in range(TopicModelObject.k):
    ngram_tokens_list = generate_n_gram(sentences, n=3, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=False)
    
    ngram_token_count = dict()
    for sentence in ngram_tokens_list:
        for feature in sentence:
            if feature in ngram_token_count.keys():
                ngram_token_count[feature] += 1
            else:
                ngram_token_count[feature] = 1    
    
    get_wordcloud(TopicModelObject, topic=i, word_count_dict = ngram_token_count)